# **Importing Libraries**

In [52]:
# For Data handling and analysis
import pandas as pd

# For Numerical calculations
import numpy as np

# For Static plots
import matplotlib.pyplot as plt

# For statistical visualizations
import seaborn as sns

# For working with Date and Time
from datetime import datetime

# For Interactive visualizations
import plotly.express as px

# For Data Split In machine learning
from sklearn.model_selection import train_test_split

# For Random Forest algorithm for regression problems
from sklearn.ensemble import RandomForestRegressor

# For evaluating the model performance
from sklearn.metrics import mean_absolute_error, r2_score

# For saving the trained model
import joblib


# **Data Loading**

In [3]:
df = pd.read_csv('uber_delhi_unclean_dataset.csv')
print("Dataset Loaded..")

Dataset Loaded..


# **Data Exploration**

**1. Dataset Shape**

In [4]:
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (2030, 10)


**2. Total Columns**

In [5]:
print(f"Total Columns:- {df.shape[1]}")

Total Columns:- 10


**3. Total Row's**

In [6]:
print(f"Total Rows:- {df.shape[0]}")

Total Rows:- 2030


**4. First 5 rows of raw data**

In [7]:
display(df.head())

,Trip_ID,Date,Time,Pickup_Location,Drop_Location,Distance_km,Payment_Type,Passenger_Count,Driver_Rating,Fare_Amount
0,UBER0001,2025-11-13,20:11,NaN,Lajpat Nagar,18.54,Cash,4,4.0,311.83
1,UBER0002,2025-07-21,07:47,Dwarka,Connaught Place,6.04,Cash,4,5.0,NaN
2,UBER0003,2025-07-26,07:01,Chandni Chowk,Indraprastha,15.01,Cash,1,4.9,209.89
3,UBER0004,2024-02-28,19:15,Chandni Chowk,Kalkaji,24.05,Wallet,1,3.1,290.55
4,UBER0005,2024-09-04,18:33,Greater Kailash,Rajouri Garden,7.40,Cash,3,3.2,145.21


**5. Basic statistics of raw data**

In [8]:
display(df.describe())

,Distance_km,Passenger_Count,Driver_Rating,Fare_Amount
count,2030.000000,2030.000000,1969.000000,1969.000000
mean,16.434956,2.514286,3.999340,276.090843
std,15.397952,1.111587,0.570671,527.509533
min,0.100000,1.000000,3.000000,1.000000
25%,8.647500,2.000000,3.500000,135.900000
50%,16.030000,3.000000,4.000000,249.210000
75%,23.105000,3.000000,4.500000,366.950000
max,400.000000,4.000000,5.000000,15000.000000


**6. Checking for missing values**

In [9]:
missing_data = df.isnull().sum()
missing_info = pd.DataFrame({
    'Missing Count': missing_data,
})
display(missing_info[missing_info['Missing Count'] > 0])

,Missing Count
Pickup_Location,62
Driver_Rating,61
Fare_Amount,61


**7. Data Types**

In [10]:
print(df.dtypes)

Trip_ID             object
Date                object
Time                object
Pickup_Location     object
Drop_Location       object
Distance_km        float64
Payment_Type        object
Passenger_Count      int64
Driver_Rating      float64
Fare_Amount        float64
dtype: object


**8. Duplicate Records**

In [11]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 27


# **Data Cleaning**

**1. Create a copy for cleaning**

In [12]:
clean_dataset = df.copy()


# **2. Handling Missing Values**

**2.1 Handle missing Pickup_Location**

In [13]:
clean_dataset['Pickup_Location'].fillna('Unknown', inplace=True)


/tmp/ipython-input-2872874659.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_dataset['Pickup_Location'].fillna('Unknown', inplace=True)


**2.2 Handle missing Driver_Rating**

In [14]:
clean_dataset['Driver_Rating'].fillna(clean_dataset['Driver_Rating'].median(), inplace=True)

/tmp/ipython-input-1550280060.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_dataset['Driver_Rating'].fillna(clean_dataset['Driver_Rating'].median(), inplace=True)


**2.2 Handle missing Fare_Amount**

In [15]:
clean_dataset['Fare_Amount'].fillna(clean_dataset['Fare_Amount'].median(), inplace=True)

/tmp/ipython-input-273633314.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_dataset['Fare_Amount'].fillna(clean_dataset['Fare_Amount'].median(), inplace=True)


# **3. Correcting Data Entry Errors**

**3.1 Fix Payment_Type spelling errors**

In [16]:
payment_corrections = {
    'Cadr': 'Card',
    'Cahs': 'Cash',
    'Walet': 'Wallet',
    'cahs': 'Cash',
    'Cadr': 'Card'
}
clean_dataset['Payment_Type'] = clean_dataset['Payment_Type'].replace(payment_corrections)

# **4. Check for unrealistic values in Distance and Fare**

**4.1 Identify potential outliers in Distance**

In [17]:
distance_outliers = clean_dataset[clean_dataset['Distance_km'] > 100]
print(f"Trips with distance > 100 km: {len(distance_outliers)}")

Trips with distance > 100 km: 5


**4.2 Identify potential outliers in Fare**

In [18]:
fare_outliers = clean_dataset[clean_dataset['Fare_Amount'] > 5000]
print(f"Trips with fare > ₹5000: {len(fare_outliers)}")

Trips with fare > ₹5000: 4


**4.3 Handle extreme outliers**

In [19]:
clean_dataset = clean_dataset[clean_dataset['Distance_km'] <= 100]
clean_dataset = clean_dataset[clean_dataset['Fare_Amount'] <= 5000]

# **5. Removing Duplicates**

In [20]:
clean_dataset = clean_dataset.drop_duplicates()

****

**Before Cleaning**

In [21]:
print(f"Remaining records: {len(df)}")

Remaining records: 2030


**After Cleaning**

In [22]:
print(f"Remaining records: {len(clean_dataset)}")

Remaining records: 1997


**Save Cleaned Data**

In [23]:
clean_dataset.to_csv('uber_delhi_clean_dataset.csv', index=False)

# **Feature Engineering**

**1. Creating a copy for feature engineering**

In [24]:
featured_dataset = clean_dataset.copy()

**2. DateTime features**

In [25]:
featured_dataset['DateTime'] = pd.to_datetime(
    featured_dataset['Date'] + ' ' + featured_dataset['Time'],
    errors='coerce'
)
featured_dataset['Hour'] = featured_dataset['DateTime'].dt.hour
featured_dataset['DayOfWeek'] = featured_dataset['DateTime'].dt.day_name()
featured_dataset['Month'] = featured_dataset['DateTime'].dt.month_name()
featured_dataset['Year'] = featured_dataset['DateTime'].dt.year
featured_dataset['DayOfMonth'] = featured_dataset['DateTime'].dt.day
featured_dataset['Weekend'] = featured_dataset['DayOfWeek'].isin(['Saturday', 'Sunday'])

**3. Time Category Features**

In [26]:
def get_time_category(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

featured_dataset['Time_Category'] = featured_dataset['Hour'].apply(get_time_category)

def get_peak_hour(hour):
    if (7 <= hour < 10) or (17 <= hour < 20):
        return 'Peak'
    else:
        return 'Off-Peak'

featured_dataset['Peak_Hour'] = featured_dataset['Hour'].apply(get_peak_hour)

**4. Trip Features**

In [27]:
featured_dataset['Fare_per_km'] = featured_dataset['Fare_Amount'] / featured_dataset['Distance_km']
featured_dataset['Fare_per_km'] = featured_dataset['Fare_per_km'].replace([np.inf, -np.inf], np.nan)
featured_dataset['Fare_per_km'].fillna(featured_dataset['Fare_per_km'].median(), inplace=True)

# Trip category based on distance
def get_trip_category(distance):
    if distance <= 5:
        return 'Short'
    elif distance <= 15:
        return 'Medium'
    else:
        return 'Long'

featured_dataset['Trip_Category'] = featured_dataset['Distance_km'].apply(get_trip_category)

# Trip efficiency (lower is better)
featured_dataset['Trip_Efficiency'] = featured_dataset['Fare_per_km'] / featured_dataset['Distance_km']

/tmp/ipython-input-2057135688.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  featured_dataset['Fare_per_km'].fillna(featured_dataset['Fare_per_km'].median(), inplace=True)


**5. Rating Features**

In [28]:
def get_rating_category(rating):
    if rating >= 4.5:
        return 'Excellent'
    elif rating >= 4.0:
        return 'Good'
    elif rating >= 3.0:
        return 'Average'
    else:
        return 'Poor'

featured_dataset['Rating_Category'] = featured_dataset['Driver_Rating'].apply(get_rating_category)

# Rating bins

featured_dataset['Rating_Bin'] = pd.cut(
    featured_dataset['Driver_Rating'],
    bins=[0, 3, 3.5, 4, 4.5, 5],
    labels=['Very Poor', 'Poor', 'Average', 'Good', 'Excellent']
)


**6. Location Features**

In [29]:
# Route feature
featured_dataset['Route'] = featured_dataset['Pickup_Location'] + ' to ' + featured_dataset['Drop_Location']

# Same location trip
featured_dataset['Same_Location_Trip'] = featured_dataset['Pickup_Location'] == featured_dataset['Drop_Location']

# Popular routes (will be calculated in analysis)
route_counts = featured_dataset['Route'].value_counts()
featured_dataset['Route_Popularity'] = featured_dataset['Route'].map(route_counts)

**7. Payment and Passenger Features**

In [30]:
# Payment type encoded
payment_mapping = {'Cash': 0, 'Card': 1, 'Wallet': 2}
featured_dataset['Payment_Type_Encoded'] = featured_dataset['Payment_Type'].map(payment_mapping)

# Passenger group
def get_passenger_group(count):
    if count == 1:
        return 'Solo'
    elif count == 2:
        return 'Couple'
    elif count == 3:
        return 'Small Group'
    else:
        return 'Large Group'

featured_dataset['Passenger_Group'] = featured_dataset['Passenger_Count'].apply(get_passenger_group)

**8.Dynamic Pricing based on Time**

In [31]:
def get_time_multiplier(hour):
    """Different pricing based on time of day"""
    if 0 <= hour < 5:      # Late Night (12AM-5AM)
        return 1.4         # 40% premium
    elif 5 <= hour < 7:    # Early Morning (5AM-7AM)
        return 1.0         # Standard
    elif 7 <= hour < 10:   # Morning Peak (7AM-10AM)
        return 1.3         # 30% premium
    elif 10 <= hour < 17:  # Day Time (10AM-5PM)
        return 1.0         # Standard
    elif 17 <= hour < 20:  # Evening Peak (5PM-8PM)
        return 1.35        # 35% premium
    elif 20 <= hour < 22:  # Early Night (8PM-10PM)
        return 1.2         # 20% premium
    else:                  # Late Night (10PM-12AM)
        return 1.4         # 40% premium

**9.Dynamic Pricing based on weekends**

In [32]:
def get_day_premium(day_name):
    """Different premium for different days"""
    day_premiums = {
        'Monday': 1.0,
        'Tuesday': 1.0,
        'Wednesday': 1.0,
        'Thursday': 1.0,
        'Friday': 1.15,    # Friday - 15% premium
        'Saturday': 1.25,  # Weekend - 25% premium
        'Sunday': 1.25     # Weekend - 25% premium
    }
    return day_premiums.get(day_name, 1.0)

**10.Adding pricing features to dataset**

In [33]:
featured_dataset['Time_Multiplier'] = featured_dataset['Hour'].apply(get_time_multiplier)
featured_dataset['Day_Premium'] = featured_dataset['DayOfWeek'].apply(get_day_premium)
featured_dataset['Is_Weekend'] = featured_dataset['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)


**11.Combined multiplier**

In [34]:
featured_dataset['Final_Multiplier'] = featured_dataset['Time_Multiplier'] * featured_dataset['Day_Premium']

**Saving Featured Dataset**

In [35]:
featured_dataset.to_csv('uber_delhi_with_features.csv', index=False)

# **Model Training**

In [36]:
# Selecting Features for Model
features = [
    'Distance_km',
    'Passenger_Count',
    'Hour',
    'Time_Multiplier',
    'Day_Premium',
    'Is_Weekend',
    'Final_Multiplier'
]

# Add individual day flags for better Understanding
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for day in days:
    featured_dataset[f'Day_{day}'] = (featured_dataset['DayOfWeek'] == day).astype(int)
    features.append(f'Day_{day}')

X = featured_dataset[features]
y = featured_dataset['Fare_Amount']

print(f"Features used: {len(features)}")
print(f"Key features: {features[:10]}...")

# splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

# Model Training
fare_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_split=5,
    random_state=42
)
fare_model.fit(X_train, y_train)

# Predictions
y_pred = fare_model.predict(X_test)

# Model performance
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: ₹{mean_absolute_error(y_test, y_pred):.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': fare_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop Feature Importance:")
print(feature_importance.head(8))



Features used: 14
Key features: ['Distance_km', 'Passenger_Count', 'Hour', 'Time_Multiplier', 'Day_Premium', 'Is_Weekend', 'Final_Multiplier', 'Day_Monday', 'Day_Tuesday', 'Day_Wednesday']...
Training set: (1597, 14), Test set: (400, 14)
R² Score: 0.8837
Mean Absolute Error: ₹35.28

Top Feature Importance:
             Feature  Importance
0        Distance_km    0.942539
2               Hour    0.017597
1    Passenger_Count    0.008905
6   Final_Multiplier    0.008867
3    Time_Multiplier    0.003407
10      Day_Thursday    0.003389
7         Day_Monday    0.002844
9      Day_Wednesday    0.002321


**Saving The Model**

In [37]:
joblib.dump(fare_model, 'uber_fare_predictor.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print("Model saved successfully!")

Model saved successfully!


# **EDA Part**

**1.Top 10 Pickup Locations**

In [38]:
fig1 = px.bar(clean_dataset['Pickup_Location'].value_counts().head(10).reset_index(name='Trips'),
              x='Pickup_Location', y='Trips',
              title='Top 10 Pickup Locations',
              labels={'Pickup_Location':'Location','Trips':'Trips'})
fig1.show()

**2.Top 10 Drop Locations**

In [39]:
fig2 = px.bar(clean_dataset['Drop_Location'].value_counts().head(10).reset_index(name='Trips'),
              x='Drop_Location', y='Trips',
              title='Top 10 Drop Locations',
              labels={'Drop_Location':'Location','Trips':'Trips'})
fig2.show()

**3.Payment Type Distribution**

In [40]:
fig3 = px.pie(clean_dataset, names='Payment_Type',
              title='Payment Type Distribution',
              hole=0.3)
fig3.show()



**4.Trip Distance Distribution**

In [41]:
fig4 = px.histogram(clean_dataset, x='Distance_km',
                    nbins=30,
                    title='Trip Distance Distribution')
fig4.show()



**5.Fare vs Distance (Trend)**

In [42]:
fig5 = px.scatter(featured_dataset, x='Distance_km', y='Fare_Amount',
                  title='Fare vs Distance Trend',
                  trendline="lowess",
                  trendline_color_override="red")

fig5.update_traces(
    marker=dict(color='blue', opacity=0.6, size=8),
    hovertemplate='Distance: %{x:.1f} km<br>Fare: ₹%{y:.2f}<extra></extra>'
)

fig5.update_layout(
    xaxis_title="Distance (km)",
    yaxis_title="Fare Amount (₹)"
)
fig5.show()

**6.Fare vs Passenger Count**

In [43]:
fig6 = px.box(clean_dataset, x='Passenger_Count', y='Fare_Amount',
              title='Fare Across Passenger Count')
fig6.show()



**7.Distance vs Driver Rating**

In [44]:
fig7 = px.scatter(clean_dataset, x='Distance_km', y='Driver_Rating',
                  hover_data=['Pickup_Location','Drop_Location','Fare_Amount'],
                  title='Distance vs Driver Rating')
fig7.show()



**8.Correlation Heatmap**

In [45]:
fig8 = px.imshow(clean_dataset[['Distance_km','Passenger_Count','Driver_Rating','Fare_Amount']].corr(),
                 text_auto=True,
                 title='Feature Correlation Heatmap')
fig8.show()



**9.Hourly distribution**

In [46]:
hourly_trips = featured_dataset['Hour'].value_counts().sort_index().reset_index()
hourly_trips.columns = ['Hour', 'Trip_Count']

fig9 = px.bar(hourly_trips, x='Hour', y='Trip_Count',
              title=' Hourly Trip Distribution',
              labels={'Hour': 'Hour of Day', 'Trip_Count': 'Number of Trips'},
              color='Trip_Count',
              color_continuous_scale='viridis')
fig9.update_traces(hovertemplate='<b>Hour: %{x}:00</b><br>Trip Count: %{y}<extra></extra>')
fig9.show()

**10.Fare Amount vs Distance**

In [47]:
fig10 = px.scatter(featured_dataset, x='Distance_km', y='Fare_Amount',
                 title='Fare Amount vs Distance',
                 color='Time_Category',
                 size='Passenger_Count',
                 hover_data=['Pickup_Location', 'Drop_Location', 'Payment_Type'],
                 labels={'Distance_km': 'Distance (km)', 'Fare_Amount': 'Fare Amount (₹)'})
fig10.update_traces(marker=dict(opacity=0.7),
                  hovertemplate='<b>Distance: %{x} km</b><br>Fare: ₹%{y}<br>Pickup: %{customdata[0]}<br>Drop: %{customdata[1]}<extra></extra>')
fig10.show()

**11.Driver Rating Distribution**

In [48]:
rating_dist = featured_dataset['Rating_Category'].value_counts().reset_index()
rating_dist.columns = ['Rating_Category', 'Count']

fig11 = px.pie(rating_dist, values='Count', names='Rating_Category',
             title='Driver Rating Distribution',
             hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdBu)
fig11.update_traces(hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>')
fig11.update_layout(showlegend=True)
fig11.show()

**12.Peak vs Off-Peak Revenue Comparison**

In [49]:
peak_revenue = featured_dataset.groupby(['Peak_Hour', 'Time_Category'])['Fare_Amount'].sum().reset_index()

fig12 = px.sunburst(peak_revenue, path=['Peak_Hour', 'Time_Category'], values='Fare_Amount',
                  title='Peak vs Off-Peak Revenue Analysis',
                  color='Peak_Hour',
                  color_discrete_map={'Peak': '#FF6B6B', 'Off-Peak': '#4ECDC4'})
fig12.update_traces(hovertemplate='<b>%{label}</b><br>Total Revenue: ₹%{value:,.0f}<extra></extra>')
fig12.show()

**13.Weekend vs Weekday Analysis**

In [50]:
weekday_analysis = featured_dataset.groupby(['DayOfWeek', 'Weekend']).agg({
    'Fare_Amount': 'mean',
    'Trip_ID': 'count'
}).reset_index()
weekday_analysis.columns = ['DayOfWeek', 'Weekend', 'Avg_Fare', 'Trip_Count']

fig13 = px.scatter(weekday_analysis, x='DayOfWeek', y='Avg_Fare',
                  size='Trip_Count', color='Weekend',
                  title='Weekend vs Weekday Analysis',
                  size_max=50,
                  color_discrete_map={True: '#FF6B6B', False: '#4ECDC4'})
fig13.update_traces(hovertemplate='<b>%{x}</b><br>Avg Fare: ₹%{y:.2f}<br>Trip Count: %{marker.size}<extra></extra>')
fig13.show()

**14.Driver Rating Distribution**

In [51]:
fig14 = px.histogram(clean_dataset, x='Driver_Rating', nbins=20,
                     title='Driver Rating Distribution', marginal='box')
fig14.show()

